# Part 3: Fabric IQ から構造化データを追加する

Zava DIY チームの製品在庫はすべて Microsoft Fabric で管理されていますが、まだ AI knowledge system にはつながっていません。サマーセールまであと 3 週間というタイミングで、それを片付けるのがあなたの仕事です。バイヤーは緊急の補充予算をどうエスカレーションすればよいかも知りたがっています。

**🎯 ミッション**
- Zava DIY の製品データに接続した **Fabric IQ Ontology knowledge source** を追加する
- 3 ソースの knowledge base（HR docs + health docs + Fabric IQ）を構築する
- 片方のサブ回答が構造化された製品在庫データから、もう片方が HR ポリシーから返る質問を投げる


## 構成要素

Part 1 と Part 2 では knowledge base がテキストドキュメントをクエリしていました。ここでは新しいソース種別である **Fabric Ontology Knowledge Source** を追加し、Microsoft Fabric の構造化された業務データに接続します。

- **Fabric Ontology Knowledge Source**: Fabric workspace の ontology に接続します。KB は実行時に `Product` などのエンティティ型に対して構造化クエリを発行します。
- **Delegated token**: このノートブックでは `InteractiveBrowserCredential` で現在のユーザーをサインインさせ、Azure AI Search の委任トークンを取得します。Search は Fabric Ontology のような保護されたソースをクエリする際に、その委任ユーザー ID を使用します。

## Step 1: 環境変数をセットアップする

Azure AI Search、Azure OpenAI、Fabric リソースの設定を読み込みます。プロジェクトフォルダーの `.env` ファイルにこのパートで必要な値が入っています。

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## Step 2: サインインと delegated token の取得

knowledge base の retrieval 呼び出しでクエリソースへのアクセスにあなたの ID を使うには、Fabric workspace へのアクセス権を持つサインイン済みユーザーの OAuth トークンが必要です。

下のセルを実行し、手順サイドバーに記載された資格情報で Azure にサインインしてください。成功すると Azure AI Search スコープの委任トークンを取得し、retrieval 時に `query_source_authorization` として使用するために保存します。

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## Step 3: 3 つの knowledge source を作成する

この knowledge base 用に、これまでと同じ 2 つの search-index ソースに加え、Fabric Ontology の knowledge source を追加して計 3 つを作成します。

* `hrdocs-knowledge-source`: `hrdocs` 検索インデックスを参照
* `healthdocs-knowledge-source`: `healthdocs` 検索インデックスを参照
* `fabric-ontology-knowledge-source`: 構造化された業務データに使用される Fabric workspace と ontology を参照

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Zava Fabric Ontology knowledge source",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")


## Step 4: マルチソース + Fabric の knowledge base を作成する

knowledge base は knowledge source 群を LLM と、retrieval の振る舞いに関する指示と組み合わせます。

このノートブックでは `outputMode` に `answerSynthesis` を指定し、アタッチしたモデルが質問に直接答えられるようにします。`retrievalReasoningEffort` は `low` に設定し、クエリプランニングと source 選択にモデルを利用します。

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-fabric-ontology-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Zava structured product data",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Fabric Ontology for structured operational data. Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 5: knowledge base にクエリを投げる

構造化された Fabric Ontology データと、アタッチした chat モデルの推論を組み合わせた質問を投げてみましょう。

knowledge base は agentic retrieval を使って、いつ search index をクエリし、いつ Fabric Ontology source をクエリするかを判断します。

⏱️ Fabric IQ knowledge source は latency が大きいため、retrieval の完了まで約 40 秒かかります。

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("I'm a Zava buyer prepping for the summer sale. "
            "Which product categories have the lowest stock levels right now? "
            "Which Zava job roles are responsible for inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            reranker_threshold=2.0
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


### activity log を確認する

今回の activity log には、Fabric IQ への呼び出しに対応する `fabricOntology` ステップと、各検索インデックスへのクエリごとの `searchIndex` ステップが表示されます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

Fabric IQ knowledge source については、references の `type` が `fabricOntology` になります。`sourceData` には、合成された回答が入った `fabricAnswer` と、構造化データの結果が入った `fabricRawData` が含まれます。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

上に表示された references を見て、次を特定してみましょう。

1. **在庫状況** に関する質問にはどの knowledge source が答えましたか？
2. **在庫と予算管理を担当するロール** に関する質問にはどの knowledge source が答えましたか？

## ✅ ミッション達成

**構築したもの:**

* ✓ **Fabric IQ Ontology Knowledge Source**: 構造化された Fabric の製品データを knowledge base に接続するソース。KB は実行時にエンティティクエリを発行できます。
* ✓ **3 ソースの Knowledge Base**: HR docs、health docs、ライブの Fabric 業務データにまたがる単一の KB。エージェントが各サブクエスチョンを適切なソースへルーティングします。
* ✓ **構造化データと非構造化データのハイブリッド取得**: 在庫状況の質問には Fabric IQ が構造化された製品エンティティから回答し、在庫と予算管理を担当するロールの質問には HR docs がドキュメントのスニペットから回答しました。

➡️ 次へ: [Part 4: Work IQ を追加する](part4-work-iq-to-kb.ipynb)。